# Silver Layer - Film Processing

This notebook processes CDC events from the Bronze layer and merges them into the Silver film table with schema evolution support and **schema drift detection**.

In [ ]:
%run ../helpers/NB_catalog_helpers

In [ ]:
%run ../helpers/NB_schema_drift_helpers

In [ ]:
%run ../helpers/NB_schema_contracts

In [ ]:
from pyspark.sql.functions import col, coalesce, expr, from_json, to_timestamp, unbase64, row_number, current_timestamp
from pyspark.sql.types import IntegerType, LongType, StringType, StructField, StructType
from pyspark.sql.window import Window
import uuid

dbutils.widgets.text("CATALOG", "workspace")
dbutils.widgets.text("BRONZE_SCHEMA", "bronze")
dbutils.widgets.text("BRONZE_TABLE", "film")
dbutils.widgets.text("SILVER_SCHEMA", "silver")
dbutils.widgets.text("SILVER_TABLE", "silver_film")
dbutils.widgets.text("CHECKPOINT_PATH", "/Volumes/workspace/default/mnt/checkpoints/film_silver")
dbutils.widgets.text("MONITORING_SCHEMA", "monitoring")
dbutils.widgets.text("SCHEMA_POLICY", "additive_only")  # strict, additive_only, permissive
dbutils.widgets.text("ALERT_CHANNEL", "log")  # log, webhook, all
dbutils.widgets.text("WEBHOOK_URL", "")  # Optional Slack/Teams webhook

CONFIG = {
    "catalog": dbutils.widgets.get("CATALOG"),
    "bronze_schema": dbutils.widgets.get("BRONZE_SCHEMA"),
    "bronze_table": dbutils.widgets.get("BRONZE_TABLE"),
    "silver_schema": dbutils.widgets.get("SILVER_SCHEMA"),
    "silver_table": dbutils.widgets.get("SILVER_TABLE"),
    "checkpoint_path": dbutils.widgets.get("CHECKPOINT_PATH"),
    "monitoring_schema": dbutils.widgets.get("MONITORING_SCHEMA"),
    "schema_policy": dbutils.widgets.get("SCHEMA_POLICY"),
    "alert_channel": dbutils.widgets.get("ALERT_CHANNEL"),
    "webhook_url": dbutils.widgets.get("WEBHOOK_URL"),
}

PIPELINE_RUN_ID = str(uuid.uuid4())

bronze_table_fqn = f"{CONFIG['catalog']}.{CONFIG['bronze_schema']}.{CONFIG['bronze_table']}"
silver_schema_fqn = f"{CONFIG['catalog']}.{CONFIG['silver_schema']}"
silver_table_fqn = f"{silver_schema_fqn}.{CONFIG['silver_table']}"
drift_table_fqn = f"{CONFIG['catalog']}.{CONFIG['monitoring_schema']}.schema_drift_log"

In [ ]:
# Initialize monitoring and Silver table
create_schema_drift_table(spark, CONFIG['catalog'], CONFIG['monitoring_schema'])
ensure_schema_exists(CONFIG['catalog'], CONFIG['silver_schema'])

existing_schema = get_existing_schema(spark, silver_table_fqn)
if existing_schema is None:
    create_silver_table_film(spark, silver_table_fqn)
else:
    print(f"Using existing Silver film table: {silver_table_fqn}")

In [ ]:
# Schema drift validation
expected_schema = get_schema_contract('silver.film')
actual_schema = get_existing_schema(spark, silver_table_fqn)

print(f"Validating schema against contract with policy: {CONFIG['schema_policy']}")

try:
    is_valid, drift_result = validate_schema_with_policy(
        spark,
        expected_schema=expected_schema,
        actual_schema=actual_schema,
        table_name=silver_table_fqn,
        drift_table=drift_table_fqn,
        source_system="postgres_cdc",
        pipeline_run_id=PIPELINE_RUN_ID,
        policy=CONFIG['schema_policy'],
        alert_channel=CONFIG['alert_channel'],
        webhook_url=CONFIG['webhook_url'] if CONFIG['webhook_url'] else None
    )
    if drift_result['has_drift']:
        print(f"⚠️ Schema drift detected: {drift_result['severity']}")
        if drift_result['added_columns']:
            print(f"  Added: {drift_result['added_columns']}")
        if drift_result['removed_columns']:
            print(f"  Removed: {drift_result['removed_columns']}")
        if drift_result['type_changes']:
            print(f"  Type changes: {drift_result['type_changes']}")
    else:
        print("✅ Schema validation passed - no drift detected")
except SchemaDriftException as e:
    print(f"🚨 PIPELINE BLOCKED: {e}")
    dbutils.notebook.exit(f"Schema drift blocked pipeline: {e}")

In [ ]:
# Read bronze stream
bronze_stream = (
    spark.readStream
    .option("mergeSchema", "true")
    .table(bronze_table_fqn)
)

# Debezium JSON payload schema for dvdrental.film
# rental_rate (NUMERIC 4,2) and replacement_cost (NUMERIC 5,2) are Debezium-encoded as {scale, value}
film_payload_schema = StructType([
    StructField("payload", StructType([
        StructField("before", StructType([
            StructField("film_id", IntegerType()),
            StructField("title", StringType()),
            StructField("description", StringType()),
            StructField("release_year", IntegerType()),
            StructField("language_id", IntegerType()),
            StructField("rental_duration", IntegerType()),
            StructField("rental_rate", StructType([
                StructField("scale", IntegerType()),
                StructField("value", StringType())
            ])),
            StructField("length", IntegerType()),
            StructField("replacement_cost", StructType([
                StructField("scale", IntegerType()),
                StructField("value", StringType())
            ])),
            StructField("rating", StringType()),
            StructField("last_update", LongType())
        ])),
        StructField("after", StructType([
            StructField("film_id", IntegerType()),
            StructField("title", StringType()),
            StructField("description", StringType()),
            StructField("release_year", IntegerType()),
            StructField("language_id", IntegerType()),
            StructField("rental_duration", IntegerType()),
            StructField("rental_rate", StructType([
                StructField("scale", IntegerType()),
                StructField("value", StringType())
            ])),
            StructField("length", IntegerType()),
            StructField("replacement_cost", StructType([
                StructField("scale", IntegerType()),
                StructField("value", StringType())
            ])),
            StructField("rating", StringType()),
            StructField("last_update", LongType())
        ])),
        StructField("op", StringType()),
        StructField("ts_ms", LongType())
    ]))
])

parsed = bronze_stream.select(
    from_json(col("value"), film_payload_schema).alias("data")
).select("data.payload.*")

In [ ]:
# Parse and transform CDC events
events = parsed.select(
    coalesce(col("after.film_id"), col("before.film_id")).alias("film_id"),
    coalesce(col("after.title"), col("before.title")).alias("title"),
    coalesce(col("after.description"), col("before.description")).alias("description"),
    coalesce(col("after.release_year"), col("before.release_year")).alias("release_year"),
    coalesce(col("after.language_id"), col("before.language_id")).alias("language_id"),
    coalesce(col("after.rental_duration"), col("before.rental_duration")).alias("rental_duration"),
    coalesce(col("after.rental_rate.value"), col("before.rental_rate.value")).alias("rental_rate_raw"),
    coalesce(col("after.rental_rate.scale"), col("before.rental_rate.scale")).alias("rental_rate_scale"),
    coalesce(col("after.length"), col("before.length")).alias("length"),
    coalesce(col("after.replacement_cost.value"), col("before.replacement_cost.value")).alias("replacement_cost_raw"),
    coalesce(col("after.replacement_cost.scale"), col("before.replacement_cost.scale")).alias("replacement_cost_scale"),
    coalesce(col("after.rating"), col("before.rating")).alias("rating"),
    coalesce(col("after.last_update"), col("before.last_update")).alias("last_update_raw"),
    col("op"),
    to_timestamp((col("ts_ms") / 1000).cast("double")).alias("event_time")
)

# Decode NUMERIC fields from base64 (Debezium encoding)
events = (
    events
    .withColumn("rental_rate", expr("cast(conv(hex(unbase64(rental_rate_raw)), 16, 10) as double) / pow(10, rental_rate_scale)").cast("decimal(4,2)"))
    .withColumn("replacement_cost", expr("cast(conv(hex(unbase64(replacement_cost_raw)), 16, 10) as double) / pow(10, replacement_cost_scale)").cast("decimal(5,2)"))
    .withColumn("last_update", to_timestamp((col("last_update_raw") / 1000000).cast("double")))
)

events = events.select(
    "film_id", "title", "description", "release_year", "language_id",
    "rental_duration", "rental_rate", "length", "replacement_cost",
    "rating", "last_update", "op", "event_time"
)

In [ ]:
def upsert_to_silver(batch_df, batch_id):
    """Upsert CDC events into Silver film table with schema evolution."""
    if not batch_df.take(1):
        return

    window = Window.partitionBy("film_id").orderBy(col("event_time").desc())
    latest = (
        batch_df
        .withColumn("rn", row_number().over(window))
        .filter(col("rn") == 1)
        .drop("rn")
    )

    existing_columns = get_existing_columns(spark, silver_table_fqn)
    core_fields = ["film_id", "title", "description", "release_year", "language_id",
                   "rental_duration", "rental_rate", "length", "replacement_cost",
                   "rating", "last_update"]
    update_set, insert_values = build_merge_clauses(
        existing_columns,
        core_fields,
        include_inserted_dt=True,
        include_updated_dt=True
    )

    delta_table = DeltaTable.forName(spark, silver_table_fqn)
    execute_merge(
        spark, delta_table, latest, update_set, insert_values,
        join_condition="t.film_id = s.film_id"
    )

In [ ]:
query = (
    events.writeStream
    .foreachBatch(upsert_to_silver)
    .option("checkpointLocation", CONFIG["checkpoint_path"])
    .option("mergeSchema", "true")
    .trigger(availableNow=True)
    .start()
)

query.awaitTermination()

print(f"✅ Silver film processing completed. Run ID: {PIPELINE_RUN_ID}")